# Multi-Sensor Fusion for Land-Cover Semantic Segmentation

This notebook coordinates the complete experimental pipeline for the study. It covers satellite data preparation, preprocessing, dataset construction, model training and evaluation for the SAR-only, Optical-only, Early Fusion, and Middle Fusion configurations. It also evaluates the U-Net and SegFormer architectures and performs paired patch-level bootstrap comparisons of model performance.

## 1. Environment Setup

The required libraries and project modules are imported, Google Drive and Google Earth Engine are initialised, and the available computational device is configured.

In [ ]:
# Import libraries used directly in the experimental workflow
import numpy as np
import torch
from torch.utils.data import DataLoader
import ee

# Import modules used throughout the experimental pipeline
from models import build_unet, MiddleFusionUNet, SegFormer
from dataset import PatchDataset, PairedDataset, train_transform
from train import train_config, train_config_dual
from bootstrap import bootstrap, bootstrap_dual, paired_bootstrap
from preprocessing import extract_and_filter_patches, file_upload, percentile_normalise
from splitting import compute_patch_proportions, compute_block_proportions, stratified_block_split, assign_patch_splits, compute_class_weights
from evaluate import evaluate_full_metrics, evaluate_full_metrics_dual, plot_confusion_matrices, plot_error_map_comparison

# Mount Google Drive to access project data, model checkpoints, and outputs
from google.colab import drive
drive.mount('/content/drive')

# Authenticate and initialise Google Earth Engine for satellite data processing
ee.Authenticate()
ee.Initialize(project='GEE_PROJECT_ID')

# Use GPU acceleration when available; otherwise run computations on the CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

## 2. Data Acquisition and Preprocessing

### 2.1 Satellite Data Preparation

The study area and acquisition period are defined before retrieving the Sentinel-1 SAR and Sentinel-2 optical imagery. The datasets are filtered to the required acquisition conditions, and median composites are created for subsequent preprocessing and model development.

In [ ]:
# Define the study area using FAO GAUL level-2 administrative boundaries
# and retain the Port Harcourt and Obio/Akpor Local Government Areas
admin_boundaries = ee.FeatureCollection('FAO/GAUL/2015/level2')
study_area = admin_boundaries.filter(
    ee.Filter.inList(
        'ADM2_NAME',
        ['Port Harcourt', 'Obio/Akpor']
    )
)

# Define the overall data acquisition period
start_date = '2024-01-01'
end_date = '2026-04-01'

# Load Sentinel-1 SAR imagery and retain observations from December to March.
# Images are restricted to IW mode, descending orbits, and VV/VH polarisation.
sentinel1_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
    .filter(ee.Filter.calendarRange(12, 3, 'month'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select(['VV', 'VH'])
)


def sentinel2_cloud_mask(image):
    """
    Applies a Scene Classification Layer (SCL) mask to Sentinel-2
    imagery and prepares the selected optical bands for analysis.

    Pixels classified as no data, saturated or defective, cloud
    shadow, or cloud are removed. The Blue, Green, Red, and NIR
    bands are retained and converted to surface reflectance values.

    Args:
        image (ee.Image): Sentinel-2 Level-2A surface reflectance image.

    Returns:
        ee.Image: Masked and scaled optical image containing B2, B3,
            B4, and B8.
    """
    scl = image.select('SCL')
    clear_mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
    )

    return (
        image
        .select(['B2', 'B3', 'B4', 'B8'])
        .updateMask(clear_mask)
        .divide(10000)  # Convert scaled integer values to surface reflectance
        .copyProperties(image, ['system:time_start'])
    )


# Load Sentinel-2 Level-2A optical imagery for December to March.
# Scenes with more than 10% reported cloud cover are excluded before
# applying the pixel-level SCL mask.
sentinel2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
    .filter(ee.Filter.calendarRange(12, 3, 'month'))
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 10))
    .map(sentinel2_cloud_mask)
)


def composite_clip(collection):
    """
    Creates a median composite from an image collection and clips
    the resulting image to the study area.

    Args:
        collection (ee.ImageCollection): Image collection to composite.

    Returns:
        ee.Image: Median composite clipped to the study area.
    """
    composite = collection.median()
    return composite.clip(study_area)


# Create the final SAR and optical median composites used as model inputs
sar_composite = composite_clip(sentinel1_collection)
optical_composite = composite_clip(sentinel2_collection)

### 2.2 Reference Label Preparation

Dynamic World is used to generate the reference labels for model training and evaluation. The most frequent class at each pixel over the study period is selected, and the original Dynamic World classes are remapped into the four target land-cover classes used in this study.

In [ ]:
# Define the four target classes used for land-cover semantic segmentation
class_names = {
    0: 'Dense Vegetation',
    1: 'Bare/Sparse Vegetation',
    2: 'Water Bodies',
    3: 'Built-up Area'
}

# Load Dynamic World reference labels for the same study period and area
dw_collection = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
    .filter(ee.Filter.calendarRange(12, 3, 'month'))
    .select('label')
)

# Report the number of Dynamic World images available after filtering
print("Dynamic World images found:", dw_collection.size().getInfo())

# Create a reference-label composite using the most frequent class at each pixel
dw_composite = dw_collection.mode().clip(study_area.geometry())

# Remap the original Dynamic World classes into the four target classes
# Dynamic World: 0=Water, 1=Trees, 2=Grass, 3=Flooded Vegetation, 4=Crops,
# 5=Shrub and Scrub, 6=Built, 7=Bare, 8=Snow and Ice
# Target: 0=Dense Vegetation, 1=Bare/Sparse Vegetation,
# 2=Water Bodies, 3=Built-up Area
remap_from = [0, 1, 2, 3, 4, 5, 6, 7]
remap_to = [2, 0, 1, 0, 1, 1, 3, 1]

labels = dw_composite.remap(remap_from, remap_to).rename('label')

### 2.3 Data Export

The prepared SAR and optical composites and the reference labels are exported from Google Earth Engine as GeoTIFF files. All outputs use a 10 m spatial resolution and the EPSG:32632 coordinate reference system, with masked pixels retained as NoData values.

In [ ]:
def export_image(image, description, folder):
    """
    Exports a Google Earth Engine image to Google Drive as a GeoTIFF.
    Masked pixels are assigned a NoData value of -9999 to preserve
    missing-data information in the exported raster.

    Args:
        image (ee.Image): Image to export.
        description (str): Export task name and output filename.
        folder (str): Destination folder in Google Drive.

    Returns:
        ee.batch.Task: Started Google Earth Engine export task.
    """
    image_with_nodata = image.unmask(-9999)

    task = ee.batch.Export.image.toDrive(
        image=image_with_nodata,
        description=description,
        folder=folder,
        scale=10,
        crs='EPSG:32632',
        region=study_area.geometry(),
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'noData': -9999}
    )

    task.start()
    return task


# Export the SAR composite, optical composite, and reference labels to Google Drive
export_image(sar_composite, 'Sentinel1_Composite', 'Dissertation_PH')
export_image(optical_composite, 'Sentinel2_Composite', 'Dissertation_PH')
export_image(labels, 'DynamicWorld_Labels', 'Dissertation_PH')

### 2.4 Data Loading and Normalisation

The exported SAR, optical, and reference-label rasters are loaded into the Python workflow. The SAR and optical bands are normalised separately using their respective percentile ranges before being combined to form the six-channel input used for the fusion experiments.

In [ ]:
# Load the exported SAR, optical, and Dynamic World GeoTIFFs from Google Drive
sar_path = "/content/drive/MyDrive/Dissertation_PH/Sentinel1_Composite.tif"
optical_path = "/content/drive/MyDrive/Dissertation_PH/Sentinel2_Composite.tif"
dynamic_world_path = "/content/drive/MyDrive/Dissertation_PH/DynamicWorld_Labels.tif"

sar, sar_profile = file_upload(sar_path)
optical, optical_profile = file_upload(optical_path)
dynamic_world_label, dw_profile = file_upload(dynamic_world_path)

# Normalise the SAR and optical bands separately using their respective
# percentile ranges, then concatenate them to create the six-channel fused input
sar_normalised = percentile_normalise(sar, ['VV', 'VH'])
optical_normalised = percentile_normalise(optical, ['Blue', 'Green', 'Red', 'NIR'])
fused = np.concatenate([sar_normalised, optical_normalised], axis=0)

# Record the number of SAR, optical, and combined input channels for model construction
n_sar_bands = sar_normalised.shape[0]  # 2 channels: VV and VH
n_optical_bands = optical_normalised.shape[0]  # 4 channels: Blue, Green, Red, and NIR
n_total_bands = n_sar_bands + n_optical_bands  # 6 channels in the fused input

## 3. Patch Extraction and Dataset Splitting

The prepared data are divided into overlapping image patches for model development. Patches containing excessive missing data are removed before the retained samples are organised into spatially separated training, validation, and test sets.

### 3.1 Patch Extraction

The six-channel fused image and corresponding reference labels are divided into overlapping 256 × 256 patches using a stride of 128 pixels. Patches containing more than 50% missing input data are discarded, and the retained fused patches are separated into their corresponding SAR and optical channels.

In [ ]:
# Extract overlapping 256 × 256 patches and discard patches with more than 50% missing data
fused_patches, label_patches, positions, stats = extract_and_filter_patches(
    fused, dynamic_world_label, patch_size=256, stride=128, nan_threshold=0.5
)

# Separate the retained fused patches into their corresponding SAR and optical channels
sar_patches = [p[:n_sar_bands] for p in fused_patches]
optical_patches = [p[n_sar_bands:] for p in fused_patches]

### 3.2 Spatial Dataset Splitting

The retained patches are grouped into 512 × 512 spatial blocks and assigned to training, validation, and test sets using multi-label stratified sampling. This spatial separation reduces overlap between the dataset splits while maintaining representation of the four target classes. Class weights are subsequently calculated using only the training data.

In [ ]:
# Compute per-patch class proportions and group patches into spatial blocks
proportions = compute_patch_proportions(label_patches, num_classes=4)
patches_df, block_props = compute_block_proportions(
    positions, proportions, block_size=512
)

# Split the spatial blocks into training, validation, and test sets
# using multi-label stratified sampling
train_blocks, val_blocks, test_blocks = stratified_block_split(
    block_props, test_size=0.15, val_size=0.15,
    class_threshold=0.005, random_state=42
)
patches_df = assign_patch_splits(
    patches_df, train_blocks, val_blocks, test_blocks
)

# Extract the patch IDs assigned to each dataset split
train_ids = patches_df[patches_df['split'] == 'train']['patch_id'].to_numpy()
val_ids = patches_df[patches_df['split'] == 'val']['patch_id'].to_numpy()
test_ids = patches_df[patches_df['split'] == 'test']['patch_id'].to_numpy()

# Compute class weights using only the training split
train_weights = compute_class_weights(
    patches_df[patches_df['split'] == 'train'], num_classes=4
)
weights_tensor = torch.tensor(train_weights, dtype=torch.float32)

### 3.3 Dataset Construction

PyTorch datasets are created for each experimental configuration using the same training, validation, and test assignments. Separate datasets are constructed for the SAR-only, optical-only, and six-channel fused inputs, while paired SAR and optical datasets are created for the dual-input Middle Fusion model. Data augmentation is applied only to the training datasets.

In [ ]:
# Create the six-channel fused datasets used for Early Fusion and the RQ2 architecture comparison
train_ds_fusion = PatchDataset(fused_patches, label_patches, train_ids, transform=train_transform)
val_ds_fusion = PatchDataset(fused_patches, label_patches, val_ids)
test_ds_fusion = PatchDataset(fused_patches, label_patches, test_ids)

# Create the single-sensor SAR datasets
train_ds_sar = PatchDataset(sar_patches, label_patches, train_ids, transform=train_transform)
val_ds_sar = PatchDataset(sar_patches, label_patches, val_ids)
test_ds_sar = PatchDataset(sar_patches, label_patches, test_ids)

# Create the single-sensor optical datasets
train_ds_optical = PatchDataset(optical_patches, label_patches, train_ids, transform=train_transform)
val_ds_optical = PatchDataset(optical_patches, label_patches, val_ids)
test_ds_optical = PatchDataset(optical_patches, label_patches, test_ids)

# Create paired SAR and optical datasets for the dual-input Middle Fusion model
train_ds_middle = PairedDataset(fused_patches, label_patches, train_ids, n_sar_bands, transform=train_transform)
val_ds_middle = PairedDataset(fused_patches, label_patches, val_ids, n_sar_bands)
test_ds_middle = PairedDataset(fused_patches, label_patches, test_ids, n_sar_bands)

## 4. RQ1: Multi-Sensor Fusion Experiments

Four input configurations are evaluated using U-Net to examine the effect of different multi-sensor fusion strategies. The experiments include two single-sensor baselines, SAR only and optical only, together with Early Fusion and Middle Fusion.


### 4.1 Model Construction

The four RQ1 models are constructed according to their respective input configurations. The SAR-only, optical-only, and Early Fusion configurations use U-Net with a ResNet34 encoder, while Middle Fusion uses separate ResNet34 encoders for the SAR and optical inputs before combining their intermediate features.

In [ ]:
# Define the directory used to store model checkpoints
CHECKPOINT_DIR = '/content/drive/MyDrive/Dissertation_PH/checkpoints'

# SAR-only baseline using a ResNet34 U-Net with the two SAR channels
model_sar = build_unet(encoder_name="resnet34", in_channels=n_sar_bands, n_classes=len(class_names))
model_sar = model_sar.to(device)

# Optical-only baseline using a ResNet34 U-Net with the four optical channels
model_optical = build_unet(encoder_name="resnet34", in_channels=n_optical_bands, n_classes=len(class_names))
model_optical = model_optical.to(device)

# Early Fusion combines the SAR and optical channels at the input of a shared ResNet34 U-Net
model_fusion = build_unet(encoder_name="resnet34", in_channels=n_total_bands, n_classes=len(class_names))
model_fusion = model_fusion.to(device)

# Middle Fusion uses separate SAR and optical encoders and combines their features at corresponding encoder stages
model_middle_fusion = MiddleFusionUNet(n_sar_bands=n_sar_bands, n_optical_bands=n_optical_bands, n_classes=len(class_names))
model_middle_fusion = model_middle_fusion.to(device)

### 4.2 Model Training

The four RQ1 configurations are trained using the same training procedure and hyperparameters to support a consistent comparison. The best checkpoint for each model is selected based on validation macro-F1.

In [ ]:
# Train the SAR-only baseline using the common training configuration
ckpt_path_sar, best_f1_sar, history_sar = train_config(
    model=model_sar,
    config_name='s1_only_res34_L6-4_WD-1e4',
    train_ds=train_ds_sar,
    val_ds=val_ds_sar,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

# Train the Optical-only baseline using the same training configuration
ckpt_path_optical, best_f1_optical, history_optical = train_config(
    model=model_optical,
    config_name='s2_only_res34_L6-4_WD-1e4',
    train_ds=train_ds_optical,
    val_ds=val_ds_optical,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

# Train Early Fusion using the six-channel fused input
ckpt_path_fusion, best_f1_fusion, history_fusion = train_config(
    model=model_fusion,
    config_name='early_fusion_res34_L6-4_WD-1e4',
    train_ds=train_ds_fusion,
    val_ds=val_ds_fusion,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

# Train Middle Fusion using the dual-input training function and paired SAR-optical datasets
ckpt_path_middle, best_f1_middle, history_middle = train_config_dual(
    model=model_middle_fusion,
    config_name='middle_fusion_res34_L6-4_WD-1e4',
    train_ds=train_ds_middle,
    val_ds=val_ds_middle,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

### 4.3 Model Evaluation

The best checkpoint for each RQ1 configuration is evaluated on the validation and held-out test sets. The same evaluation procedure is used across all four configurations, with the dual-input evaluation function used for Middle Fusion.

In [ ]:
# Create validation DataLoaders for the four RQ1 configurations
val_loader_sar = DataLoader(val_ds_sar, batch_size=8, shuffle=False)
val_loader_optical = DataLoader(val_ds_optical, batch_size=8, shuffle=False)
val_loader_fusion = DataLoader(val_ds_fusion, batch_size=8, shuffle=False)
val_loader_middle = DataLoader(val_ds_middle, batch_size=8, shuffle=False)

# Load the best checkpoint for each configuration and evaluate on the validation set

# SAR-only baseline
model_sar.load_state_dict(torch.load(ckpt_path_sar))
val_results_sar, val_f1_pc_sar, val_prec_pc_sar, val_recall_pc_sar, val_cm_sar = evaluate_full_metrics(
    model_sar, val_loader_sar, len(class_names), class_names, device
)

# Optical-only baseline
model_optical.load_state_dict(torch.load(ckpt_path_optical))
val_results_optical, val_f1_pc_optical, val_prec_pc_optical, val_recall_pc_optical, val_cm_optical = evaluate_full_metrics(
    model_optical, val_loader_optical, len(class_names), class_names, device
)

# Early Fusion
model_fusion.load_state_dict(torch.load(ckpt_path_fusion))
val_results_fusion, val_f1_pc_fusion, val_prec_pc_fusion, val_recall_pc_fusion, val_cm_fusion = evaluate_full_metrics(
    model_fusion, val_loader_fusion, len(class_names), class_names, device
)

# Middle Fusion uses the dual-input evaluation function
model_middle_fusion.load_state_dict(torch.load(ckpt_path_middle))
val_results_middle, val_f1_pc_middle, val_prec_pc_middle, val_recall_pc_middle, val_cm_middle = evaluate_full_metrics_dual(
    model_middle_fusion, val_loader_middle, len(class_names), class_names, device
)


# Create test DataLoaders for the four RQ1 configurations
test_loader_sar = DataLoader(test_ds_sar, batch_size=8, shuffle=False)
test_loader_optical = DataLoader(test_ds_optical, batch_size=8, shuffle=False)
test_loader_fusion = DataLoader(test_ds_fusion, batch_size=8, shuffle=False)
test_loader_middle = DataLoader(test_ds_middle, batch_size=8, shuffle=False)

# Load the best checkpoint for each configuration and evaluate on the test set

# SAR-only baseline
model_sar.load_state_dict(torch.load(ckpt_path_sar))
results_sar, f1_pc_sar, prec_pc_sar, recall_pc_sar, cm_sar = evaluate_full_metrics(
    model_sar, test_loader_sar, len(class_names), class_names, device
)

# Optical-only baseline
model_optical.load_state_dict(torch.load(ckpt_path_optical))
results_optical, f1_pc_optical, prec_pc_optical, recall_pc_optical, cm_optical = evaluate_full_metrics(
    model_optical, test_loader_optical, len(class_names), class_names, device
)

# Early Fusion
model_fusion.load_state_dict(torch.load(ckpt_path_fusion))
results_fusion, f1_pc_fusion, prec_pc_fusion, recall_pc_fusion, cm_fusion = evaluate_full_metrics(
    model_fusion, test_loader_fusion, len(class_names), class_names, device
)

# Middle Fusion uses the dual-input evaluation function
model_middle_fusion.load_state_dict(torch.load(ckpt_path_middle))
results_middle, f1_pc_middle, prec_pc_middle, recall_pc_middle, cm_middle = evaluate_full_metrics_dual(
    model_middle_fusion, test_loader_middle, len(class_names), class_names, device
)

### 4.4 Statistical Comparison

Paired patch-level bootstrap resampling is used to compare Early Fusion with the other RQ1 configurations on the held-out test set. For each comparison, the same test patches are resampled for both models across 5,000 bootstrap replicates, and the difference in macro-F1 is evaluated using a 95% confidence interval.

In [ ]:
# Collect patch-level predictions for the four RQ1 configurations
preds_sar, labels_sar = bootstrap(model_sar, test_ds_sar, device)
preds_optical, labels_optical = bootstrap(model_optical, test_ds_optical, device)
preds_fusion, labels_fusion = bootstrap(model_fusion, test_ds_fusion, device)
preds_middle, labels_middle = bootstrap_dual(model_middle_fusion, test_ds_middle, device)

# Compare Early Fusion with each alternative using paired patch-level
# bootstrap resampling with 5,000 replicates and 95% confidence intervals
paired_bootstrap(
    preds_fusion, preds_sar, labels_fusion,
    "Early Fusion", "SAR-only",
    n_iterations=5000, lower=2.5, upper=97.5
)

paired_bootstrap(
    preds_fusion, preds_optical, labels_fusion,
    "Early Fusion", "Optical-only",
    n_iterations=5000, lower=2.5, upper=97.5
)

paired_bootstrap(
    preds_fusion, preds_middle, labels_fusion,
    "Early Fusion", "Middle Fusion",
    n_iterations=5000, lower=2.5, upper=97.5
)

## 5. RQ2: Architecture Comparison

The best-performing fusion configuration from RQ1, Early Fusion, is used to compare two different approaches to learning spatial information: U-Net using a ResNet34 encoder and SegFormer using a MiT-B2 encoder. Both architectures use the same six-channel fused input, training procedure, and dataset splits. SegFormer is trained from scratch and evaluated on the validation and test sets before its test performance is statistically compared with U-Net using paired patch-level bootstrap resampling.

In [ ]:
# Build the SegFormer architecture using the MiT-B2 encoder and All-MLP decoder head
model_segformer = SegFormer(encoder_name="mit_b2", in_channels=n_total_bands, n_classes=len(class_names))
model_segformer = model_segformer.to(device)

# Train SegFormer using the same six-channel fused datasets and training configuration
ckpt_path_segformer, best_f1_segformer, history_segformer = train_config(
    model=model_segformer,
    config_name='segformer_mitb2_L6-4_WD-1e4',
    train_ds=train_ds_fusion,
    val_ds=val_ds_fusion,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

# Load the best SegFormer checkpoint selected using validation macro-F1
model_segformer.load_state_dict(torch.load(ckpt_path_segformer))

# Evaluate SegFormer on the validation set
val_results_segformer, val_f1_pc_segformer, val_prec_pc_segformer, val_recall_pc_segformer, val_cm_segformer = evaluate_full_metrics(
    model_segformer, val_loader_fusion, len(class_names), class_names, device
)

# Evaluate SegFormer on the held-out test set
results_segformer, f1_pc_segformer, prec_pc_segformer, recall_pc_segformer, cm_segformer = evaluate_full_metrics(
    model_segformer, test_loader_fusion, len(class_names), class_names, device
)

# Collect patch-level SegFormer predictions for statistical comparison
preds_segformer, labels_segformer = bootstrap(model_segformer, test_ds_fusion, device)

# Compare SegFormer with the Early Fusion U-Net using paired patch-level
# bootstrap resampling with 5,000 replicates and a 95% confidence interval
paired_bootstrap(
    preds_segformer, preds_fusion, labels_segformer,
    "SegFormer", "U-Net",
    n_iterations=5000, lower=2.5, upper=97.5
)

## 6. Results Visualisation

The performance of U-Net and SegFormer is examined visually using normalised confusion matrices and pixel-level prediction outputs. A representative test patch is used to compare the reference labels with the predictions produced by both architectures and to visualise their classification errors.

In [ ]:
# Plot normalized confusion matrices for the Early Fusion U-Net and SegFormer 
plot_confusion_matrices(
    cm_fusion, cm_segformer, class_names,
    "(a) U-Net (ResNet34)", "(b) SegFormer",
    normalize=True
)

# Generate predictions for a representative test patch for qualitative comparison (Figure 4.3)
idx = 5
img, true_label = test_ds_fusion[idx]

with torch.no_grad():
    pred_unet = model_fusion(img.unsqueeze(0).to(device)).argmax(dim=1).cpu().squeeze().numpy()
    pred_segformer = model_segformer(img.unsqueeze(0).to(device)).argmax(dim=1).cpu().squeeze().numpy()

# Compare the reference labels with U-Net and SegFormer predictions and their error maps
plot_error_map_comparison(
    true_label.numpy(), pred_unet, pred_segformer, class_names,
    "U-Net", "SegFormer"
)